<a href="https://colab.research.google.com/github/Architag1503/Colab/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
# Import necessary layers for building the neural network
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN , Dense

In [2]:
sentences = [
 # Positive sentences for sentiment analysis
 "I love this product",
 "This movie made me smile",
 "Service was friendly and quick",
 "Today felt bright and happy",
 "This is the best day",
 "Absolutely fantastic experience",
 "I enjoyed every single moment",
 "Great job, well done",
 "The food tasted delicious",
 "Totally recommend to everyone",
 "Very satisfied with results",
 "This worked better than expected",
 "Amazing quality and value",
 "Such a pleasant surprise",
 "I feel positive about this",
 # Negative sentences for sentiment analysis
 "I hate this product",
 "This movie bored me",
 "Service was rude and slow",
 "Today was cold and lonely",
 "This is the worst day",
 "Terrible experience overall",
 "I regret buying this",
 "Very disappointed with results",
 "The food tasted awful",
 "Do not recommend this",
 "It broke after one use",
 "Not worth the money",
 "Utterly frustrating and annoying",
 "I feel negative about this",
 "Such a waste of time",
]
# Assign labels: 1 for positive sentiment, 0 for negative sentiment
# The first 15 sentences are positive, the next 15 are negative
labels = [1]*15 + [0]*15
labels = np.array(labels)

In [3]:
# Define the vocabulary size for the tokenizer
vocab_size = 2000
# Initialize the Tokenizer to vectorize the text
# oov_token='<OOV>' handles out-of-vocabulary words
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
# Fit the tokenizer on the training sentences to build the vocabulary
tokenizer.fit_on_texts(sentences)
# Convert sentences to sequences of integers (word indices)
seqs = tokenizer.texts_to_sequences(sentences)
# Determine the maximum sequence length for padding
max_len = max([len(seq) for seq in seqs])
# Pad sequences to ensure uniform length for neural network input
# 'post' padding adds zeros at the end of sequences
X = pad_sequences(seqs, maxlen=max_len, padding='post')
# Assign the numerical labels to y
y = labels

In [4]:
# Display the first processed sequence (sentence) to see its numerical representation
X[0]

array([ 3, 26,  2,  7,  0], dtype=int32)

In [5]:
# Display the last processed sequence (sentence) to see its numerical representation
X[29]

array([21, 22, 85, 86, 87], dtype=int32)

In [6]:
# Define the dimensionality of the word embeddings
embed_dim = 16
# Define the number of units (neurons) in the SimpleRNN layer
rnn_units = 8

In [7]:
# Define the input layer with a shape equal to the maximum sequence length
inp = Input(shape=(max_len,) , dtype="int32" , name="input")
# Add an Embedding layer to convert integer indices into dense vectors
# mask_zero=True ignores padding (0s) in the input sequences
x = Embedding(input_dim=vocab_size, output_dim=embed_dim , mask_zero=True , name='embed')(inp)
# Add a SimpleRNN layer with specified units
# return_sequences=True ensures the RNN outputs a sequence, not just the last state
rnn = SimpleRNN(units = rnn_units , return_sequences=True, return_state=False , name="simple_rnn")
x_all = rnn(x)
# Extract the last hidden state from the sequence output of the RNN
# This represents the context of the entire sequence for classification
x_last = x_all[:,-1,:]
# Add a Dense output layer with a single unit and sigmoid activation for binary classification
out = Dense(1 , activation="sigmoid" , name="output")(x_last)
# Create the Keras Model, specifying inputs and outputs
model = Model(inputs=inp , outputs=out)
# Compile the model with Adam optimizer, binary cross-entropy loss (for binary classification),
# and accuracy as the evaluation metric
model.compile(optimizer='adam' , loss='binary_crossentropy' , metrics=['accuracy'])
# Print a summary of the model's architecture
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 5)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 5, 16)     │     32,000 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 5)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 5, 8)      │        200 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 8)         │          0 │ simple_rnn[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │          9 │ get_item[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# Train the model using the prepared data
# X: input sequences, y: corresponding labels
# epochs: number of training iterations
# batch_size: number of samples per gradient update
# verbose=1: display progress bar during training
model.fit(X , y , epochs=25, batch_size = 8 , verbose = 1)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.6333 - loss: 0.6889
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6667 - loss: 0.6750
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7667 - loss: 0.6622
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8667 - loss: 0.6501
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9000 - loss: 0.6366
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9333 - loss: 0.6242
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9667 - loss: 0.6097
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9667 - loss: 0.5961
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9667 - loss: 0.5802
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.5638
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.5471
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.5289
E

In [9]:
# Create an intermediate model to extract outputs from specific layers
# This model will output the embeddings and the hidden states from the SimpleRNN layer
intermediate_model = Model(inputs=model.input, outputs=[model.get_layer('embed').output, model.get_layer('simple_rnn').output])
# Display the intermediate model object
intermediate_model

<Functional name=functional_1, built=True>

In [10]:
# Use the intermediate model to predict and get the outputs of the embedding and RNN layers
# pre_timestamps will hold the embedding outputs for each word in each sequence
# hidden_states will hold the hidden states of the RNN for each timestep in each sequence
pre_timestamps, hidden_states = intermediate_model.predict(X)

# Print the shapes of the extracted outputs to understand their dimensions
print(f"Shape of pre-timestamps (embedding output): {pre_timestamps.shape}")
print(f"Shape of hidden states (RNN output): {hidden_states.shape}")

# Display the embedding output for the first sentence
print("\nPre-timestamps (embedding output) for the first sentence:\n", pre_timestamps[0])
# Display the RNN hidden states for the first sentence
print("\nHidden states (RNN output) for the first sentence:\n", hidden_states[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
Shape of pre-timestamps (embedding output): (30, 5, 16)
Shape of hidden states (RNN output): (30, 5, 8)

Pre-timestamps (embedding output) for the first sentence:
 [[-0.00703677  0.01857804  0.07140633  0.01040449 -0.00868529  0.04248546
   0.04006826  0.0390295  -0.02264495  0.05833784 -0.04859962  0.04313717
  -0.02963844  0.03824937  0.03164714  0.02122192]
 [-0.09590503  0.05832615 -0.02122229  0.06259802  0.09151059  0.09002747
   0.08176088  0.0626601   0.09974363 -0.08782245 -0.01342807  0.08169588
  -0.04486599 -0.09735032  0.04809386  0.02223106]
 [-0.11268502 -0.04637536  0.0177887   0.02761975 -0.06026914 -0.05085479
   0.01433167 -0.01186466 -0.0325049  -0.02943572  0.04283499  0.03957059
  -0.00366499  0.01586414  0.02778274  0.03004771]
 [ 0.01776768  0.0137767  -0.03461123  0.0232106  -0.02128789 -0.02462059
   0.00138649  0.0407146  -0.00409515  0.04054777  0.00455179 -0.02880089
   0.02794337 -0.03483172 -0.03639805 -0.03233923]
 